# Python Day 4 실습 워크시트 — NumPy·pandas 입문 및 미니 프로젝트

**HYUNDAI AI Insight Campus 온디바이스 AI · 프로그래밍 언어 · 9/7(월)**

## 사용 방법 (Day 1~3과 동일)
1. 파일 › **드라이브에 사본 저장** → `Day4_이름.ipynb`
2. 맨 아래 **⚙️ 설정** 셀을 먼저 실행 (스트림 데이터·parse_frame 내장)
3. 교시별 **따라하기 → TODO → ✅ 자동 채점 → 🔍 자기 점검**
4. 6교시 **프로젝트 필수** 채점까지 완료 후 제출 요약 실행
5. 애니메이션: day4 폴더 `index.html` (D4-1 · D4-2 + 재사용)

## ⚙️ 설정 (가장 먼저 실행)

In [ ]:
#@title ⚙️ 설정 — 이름을 입력하고 실행하세요
name = ""  #@param {type:"string"}

import hashlib, math, random, os, sys, json, csv, time, io, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import clear_output
SCORES = {}
def _h(v): return hashlib.md5(repr(v).encode()).hexdigest()[:6]
def _check(section, tests):
    ok = 0
    for label, fn, hint in tests:
        try:
            if fn(): ok += 1; print(f"✅ {label}")
            else: print(f"❌ {label} — {hint}")
        except NameError as e: print(f"❌ {label} — 변수/함수가 없습니다: {e}")
        except NotImplementedError: print(f"❌ {label} — 아직 작성하지 않았습니다")
        except Exception as e: print(f"❌ {label} — {type(e).__name__}: {e}")
    SCORES[section] = (ok, len(tests))
    print(f"\nPASS {ok}/{len(tests)}")
def _quiz(section, qid, answer, correct, why):
    if answer == "선택": print("먼저 답을 고르세요."); return
    ok = (answer == correct)
    SCORES.setdefault(f"{section}-quiz", {})[qid] = ok
    print(("✅ 정답. " if ok else f"❌ 오답 (정답: {correct}). ") + why)

# ── Day 1~3에서 만든 도구 (제공) ──
class FrameError(Exception):
    pass

def parse_frame(line):
    """$N=..,T=..,H=..,D=..* → (node_id, float, int, int). 검증 실패 시 FrameError."""
    line = line.strip()
    if not (line.startswith("$") and line.endswith("*")):
        raise FrameError(f"손상 프레임: {line!r}")
    d = {}
    for f in line[1:-1].split(","):
        k, v = f.split("=")
        d[k] = v
    return (d["N"], float(d["T"]), int(d["H"]), int(d["D"]))

# ── 데이터 준비 (data/ 폴더에 스트림 파일 생성) ──
os.makedirs("data", exist_ok=True); os.makedirs("out", exist_ok=True)
_BOOT = """$N=ESP32-01,T=26.2,H=69,D=220*
$N=ESP32-02,T=32.2,H=70,D=130*
$N=ESP32-03,T=22.4,H=42,D=220*
$N=ESP32-01,T=26.5,H=73,D=103*
$N=ESP32-02,T=30.0,H=41,D=220*
$N=ESP32-03,T=22.8,H=54,D=168*
$N=ESP32-01,T=25.8,H=57,D=220*
$N=ESP32-02,T=38.3,H=54,D=108*
$N=ESP32-03,T=22.4,H=46,D=172*
$N=ESP32-01,T=27.4,H=40,D=106*
$N=ESP32-02,T=40.9,H=65,D=142*
$N=ESP32-03,T=25.1,H=57,D=124*
$N=ESP32-01,T=27.9,H=47,D=108*

$N=ESP32-03,T=24.0,H=48,D=220*
$N=
$N=ESP32-02,T=30.6,H=46,D=113*
$N=ESP32-03,T=22.0,H=61,D=172*
$N=ESP32-0
$N=ESP32-02,T=31.3,H=48,D=146*
$N=ESP32-03,T=25.6,H=55,D=143*
$N=ESP32-01,T=24.8,H=64,D=220*
$N=ESP32-02,T=29.4,H=42,D=129*
$N=ESP32-03,T=25.3,H=71,D=220*
$N=ESP32-01,T=27.9,H=48,D=186*
$N=ESP32-02,T=31.5,H=44,D=117*
$N=ESP32-03,T=22.3,H=66,D=110*
$N=ESP32-0
$N=ESP32-02,T=31.6,H=73,D=160*
$N=ESP32-03,T=23.9,H=62,D=105*"""
_D4 = """$N=ES
$N=ESP32-02,T=29.4,H=70,D=129*
$N=ESP32-03,T=24.3,H=51,D=135*
$N=ESP32-01,T=25.2,H=57,D=121*
$N=ESP32-02,T=32.6,H=53,D=118*
$N=ESP
$N=ESP32-01,T=25.2,H=73,D=201*
$N=ESP32-02,T=30.7,H=53,D=134*
$N=ESP32-03,T=24.0,H=52,D=122*
NESP32-01,T=27.9,H=70,D=112
$N=ESP32-02,T=32.2,H=42,D=220*
$N=ESP32-03,T=23.1,H=52,D=118*
$N=ESP32-01,T=26.5,H=45,D=174*
$N=ESP32-02,T=30.6,H=48,D=129*
$N=ESP32-03,T=22.8,H=70,D=155*
NESP32-01,T=24.5,H=66,D=195
$N=ESP32-02,T=2X.5,H=69,D=220*
$N=ESP32-03,T=22.8,H=70,D=220*
$N=ESP32-01,T=2X.5,H=72,D=128*
$N=ESP32-02,T=32.2,H=42,D=160*
$N=ESP32-03,T=22.1,H=59,D=133*
$N=ESP32-01,T=27.5,H=40,D=143*
$N=ESP32-02,T=29.1,H=71,D=220*
$N=ESP32-03,T=25.2,H=40,D=220*
$N=ESP32-01,T=24.8,H=69,D=122*
$N=ESP32-02,T=32.8,H=70,D=132*
$N=ESP32-03,T=24.3,H=62,D=215*
$N=ESP32-01,T=26.9,H=72,D=187*
$N=ESP32-02,T=30.8,H=51,D=205*
$N=ESP32-03,T=25.0,H=69,D=143*
$N=ESP32-01,T=25.1,H=45,D=220*
$N=ESP32-02,T=31.2,H=57,D=130*
$N=ESP32-03,T=24.4,H=49,D=117*
$N=ESP32-01,T=24.5,H=59,D=109*
$N=ESP32-02,T=31.2,H=65,D=216*
$N=ESP32-03,T=25.8,H=66,D=220*
$N=ESP32-01,T=26.1,H=74,D=150*
$N=ESP32-02,T=31.2,H=50,D=220*
$N=ESP32-03,T=23.7,H=57,D=136*
$N=ESP32-01,T=26.5,H=60,D=114*
$N=ESP32-02,T=29.6,H=65,D=138*
$N=ESP32-03,T=2X.5,H=63,D=220*
$N=ESP32-01,T=26.9,H=53,D=122*
$N=ESP32-02,T=32.9,H=40,D=158*
$N=ESP32-03,T=24.7,H=44,D=121*
$N=ESP32-01,T=24.2,H=67,D=220*
$N=ESP32-02,T=30.6,H=62,D=186*
$N=ESP32-03,T=22.5,H=63,D=187*
$N=ESP32-01,T=24.5,H=57,D=200*
$N=ESP32-02,T=29.4,H=59,D=104*
$N=ESP32-03,T=22.1,H=75,D=127*
$N=ESP32-01,T=24.0,H=46,D=144*
$N=ESP32-02,T=32.0,H=72,D=109*
$N=ESP32-03,T=23.0,H=56,D=203*
$N=ES
$N=ESP32-02,T=31.6,H=65,D=220*
$N=ESP32-03,T=24.0,H=52,D=184*
$N=ESP32-01,T=24.5,H=48,D=142*
$N=
$N=ESP32-03,T=23.8,H=40,D=220*
$N=ESP32-01,T=25.7,H=69,D=219*
$N=ESP32-02,T=32.7,H=52,D=220*
$N=ESP32-03,T=24.4,H=60,D=111*
$N=ESP32-01,T=25.8,H=61,D=132*
$N=ESP32-02,T=31.2,H=59,D=135*
$N=ESP32-03,T=24.5,H=48,D=164*
$N=ESP32-01,T=24.8,H=71,D=190*
$N=ESP32-02,T=30.2,H=46,D=208*
$N=ESP32-03,T=22.6,H=64,D=220*
$N=ESP32-01,T=24.4,H=53,D=155*
$N=ESP32-02,T=32.4,H=45,D=172*
$N=ESP32-03,T=22.5,H=72,D=110*
$N=ESP32-01,T=26.6,H=67,D=157*
$N=ESP32-02,T=30.1,H=68,D=131*
$N=ESP32-03,T=23.4,H=42,D=171*
$N=ESP32-01,T=24.7,H=54,D=124*
$N=ESP32-02,T=30.2,H=51,D=135*
$N=ESP32-03,T=2X.5,H=64,D=138*
$N=ESP32-01,T=26.7,H=54,D=124*
$N=ESP32-02,T=31.0,H=46,D=137*
$N=ESP32-03,T=25.9,H=50,D=131*
$N=ESP32-01,T=24.4,H=68,D=112*
$N=ESP32-02,T=30.0,H=45,D=118*
$N=ESP32-03,T=22.3,H=40,D=168*
$N=ESP32-01,T=26.4,H=67,D=139*
$N=ESP32-02,T=31.4,H=51,D=194*
$N=ESP32-03,T=22.8,H=71,D=220*
$N=ESP32-01,T=26.2,H=55,D=112*
$N=ESP32-02,T=32.4,H=67,D=220*
$N=ESP32-03,T=24.0,H=68,D=134*
$N=ESP32-01,T=25.9,H=64,D=220*
$N=ESP3
$N=ESP32-03,T=24.1,H=51,D=106*
$N=ESP32-01,T=24.7,H=43,D=124*
$N=ESP32-02,T=31.8,H=69,D=205*
$N=ESP32-03,T=24.2,H=48,D=220*
$N=ESP32-01,T=26.5,H=58,D=106*
$N=ESP32-02,T=31.6,H=64,D=140*
$N=ESP32-03,T=24.8,H=43,D=185*

$N=ESP32-02,T=31.2,H=58,D=181*
$N=ESP32-03,T=25.0,H=52,D=114*
$N=ESP32-01,T=24.5,H=74,D=216*
$N=ESP32-02,T=31.9,H=72,D=220*
$N=ESP32-03,T=24.2,H=47,D=108*
$N=ESP32-01,T=24.1,H=74,D=133*
$N=ESP32-02,T=32.1,H=46,D=126*
$N=ESP32-03,T=24.5,H=68,D=109*
$N=ESP32-01,T=26.1,H=58,D=220*
$N=ESP32-02,T=29.9,H=51,D=200*
$N=ESP32-03,T=23.6,H=74,D=143*
$N=ESP32-01,T=24.2,H=58,D=124*
$N=ESP32-02,T=30.1,H=70,D=129*
$N=ESP32-03,T=25.7,H=73,D=197*
$N=ESP32-01,T=27.6,H=58,D=153*
$N=ESP32-02,T=32.8,H=40,D=119*
$N=ESP32-03,T=24.9,H=59,D=219*
$N=ESP32-01,T=25.6,H=48,D=220*
$N=ESP32-02,T=29.9,H=71,D=182*
$N=ESP32-03,T=22.7,H=62,D=146*
$N=ESP32-01,T=25.7,H=56,D=189*
$N=ESP32-02,T=30.1,H=53,D=220*
$N=ESP32-03,T=24.5,H=59,D=127*
$N=ESP32-01,T=25.1,H=43,D=206*
$N=ESP32-02,T=30.5,H=46,D=172*
$N=ESP32-03,T=24.8,H=51,D=107*
$N=ESP32-01,T=25.0,H=54,D=136*
$N=ESP32-02,T=32.5,H=53,D=134*
$N=ESP32-03,T=22.5,H=68,D=220*
$N=ESP32-01,T=25.6,H=48,D=206*
$N=ESP32-02,T=30.6,H=41,D=185*
$N=ESP32-03,T=23.6,H=45,D=137*
$N=ESP32-01,T=25.2,H=56,D=131*
$N=ESP32-02,T=29.4,H=68,D=113*
$N=ESP32-03,T=25.1,H=53,D=206*
$N=ESP32-01,T=24.5,H=43,D=113*
$N=ESP32-02,T=31.7,H=46,D=140*
$N=ESP32-03,T=25.1,H=75,D=124*
$N=ESP32-01,T=24.4,H=49,D=220*
$N=ESP32-02,T=30.5,H=66,D=204*
$N=ESP32-03,T=24.9,H=53,D=201*
$N=ESP32-01,T=25.2,H=57,D=142*
$N=ESP32-02,T=32.3,H=44,D=170*
$N=ESP32-03,T=22.8,H=49,D=140*
$N=ESP32-01,T=27.1,H=74,D=120*
$N=ESP32-02,T=30.8,H=72,D=126*
$N=ESP32-03,T=25.4,H=74,D=220*
$N=ESP32-01,T=27.3,H=48,D=203*
$N=ESP32-02,T=29.4,H=74,D=220*
$N=ESP32-03,T=22.5,H=52,D=189*
$N=ESP32-01,T=24.4,H=44,D=131*
$N=ESP32-02,T=29.5,H=60,D=220*
$N=ESP32-03,T=22.4,H=67,D=118*
$N=ESP32-01,T=24.6,H=46,D=195*
$N=ESP32-02,T=32.6,H=40,D=175*
$N=ESP32-03,T=25.7,H=69,D=145*
$N=ESP32-01,T=26.8,H=57,D=115*
$N=ESP32-02,T=29.6,H=52,D=220*
$N=ESP32-03,T=25.7,H=56,D=122*
$N=ESP3
$N=ESP32-02,T=32.1,H=72,D=158*
$N=ESP32-03,T=23.2,H=40,D=205*
$N=ESP32-01,T=27.7,H=44,D=220*
$N=ESP32-02,T=30.8,H=52,D=122*
$N=ESP32-03,T=24.5,H=72,D=111*
$N=ESP32-01,T=25.9,H=66,D=128*
$N=ESP32-02,T=30.7,H=62,D=176*
$N=ESP32-03,T=24.1,H=45,D=103*
$N=ESP32-01,T=27.5,H=73,D=220*

$N=ESP32-03,T=24.0,H=68,D=142*
$N=ESP32-01,T=26.7,H=68,D=130*
$N=ESP32-02,T=31.8,H=75,D=125*
$N=ESP32-03,T=23.8,H=73,D=112*
$N=ESP32-01,T=25.2,H=56,D=127*
$N=ESP32-02,T=30.3,H=61,D=122*
$N=ESP32-03,T=23.3,H=40,D=220*
$N=ESP32-01,T=25.1,H=67,D=126*
$N=ESP32-02,T=2X.5,H=44,D=118*
$N=ESP32-
$N=ESP32-01,T=25.8,H=52,D=170*
$N=ESP32-02,T=29.7,H=71,D=220*
$N=ESP32-03,T=2X.5,H=50,D=132*
$N=ESP32-01,T=25.1,H=47,D=146*
$N=ESP32-02,T=31.2,H=67,D=220*
$N=ESP32-03,T=24.8,H=71,D=220*
$N=ESP32-01,T=26.0,H=47,D=131*
$N=ESP32-02,T=30.7,H=69,D=220*
$N=ESP32-03,T=22.1,H=45,D=220*
$N=ESP32-01,T=25.6,H=75,D=134*
$N=ESP32-02,T=29.3,H=69,D=123*
$N=ESP32-03,T=25.5,H=53,D=100*
$N=ESP32-01,T=26.4,H=58,D=113*
$N=ESP32-02,T=31.9,H=55,D=220*
$N=ESP32-03,T=24.4,H=47,D=220*
$N=ESP
$N=ESP32-02,T=29.4,H=57,D=129*
$N=ESP32-03,T=25.1,H=54,D=164*
$N=ESP32-01,T=25.8,H=72,D=143*
NESP32-02,T=30.9,H=60,D=220
$N=ESP32-03,T=25.3,H=48,D=191*
$N=ESP32-01,T=25.5,H=42,D=104*
$N=ESP32-02,T=38.3,H=59,D=200*
$N=ESP32-03,T=24.7,H=75,D=220*

$N=ESP32-02,T=37.6,H=60,D=128*
$N=ESP32-03,T=25.4,H=42,D=191*
$N=ESP32-01,T=26.4,H=52,D=151*
$N=ESP32-02,T=40.8,H=53,D=220*
$N=ESP32-03,T=25.8,H=51,D=104*
$N=ESP32-01,T=27.6,H=48,D=158*
$N=ESP32-02,T=39.4,H=66,D=220*

$N=ESP32-01,T=2X.5,H=53,D=220*
$N=ESP32-02,T=40.8,H=55,D=139*
$N=ESP32-03,T=23.1,H=48,D=220*
$N=ESP32-01,T=24.3,H=62,D=135*
$N=ESP32-02,T=39.5,H=74,D=166*
$N=ESP32-03,T=24.0,H=41,D=105*
$N=ESP32-01,T=27.7,H=67,D=210*

$N=ESP32-03,T=24.8,H=58,D=103*
$N=ESP32-01,T=24.7,H=60,D=220*
$N=ESP32-02,T=39.5,H=63,D=120*
$N=ESP32-03,T=22.0,H=58,D=139*
$N=ESP32-01,T=26.9,H=50,D=141*
$N=ESP32-02,T=39.8,H=53,D=220*
$N=ESP32-03,T=24.5,H=50,D=115*
$N=ESP32-01,T=26.6,H=41,D=115*
$N=ESP32-02,T=37.7,H=63,D=133*
$N=ESP32-03,T=22.1,H=49,D=182*
$N=ESP32-01,T=26.5,H=48,D=220*
$N=ESP32-02,T=37.8,H=52,D=124*
$N=ESP32-03,T=22.2,H=66,D=220*
$N=ESP32-01,T=25.9,H=67,D=220*
$N=ESP32-02,T=37.9,H=75,D=218*
$N=ESP32-03,T=24.3,H=75,D=220*
$N=ESP32-01,T=25.5,H=52,D=104*
$N=ESP32-02,T=38.9,H=62,D=132*


$N=ESP32-02,T=40.7,H=68,D=193*

$N=ESP32-01,T=24.8,H=64,D=220*
$N=ESP32-02,T=38.0,H=44,D=121*

NESP32-01,T=25.5,H=42,D=108
$N=ESP32-02,T=38.9,H=61,D=113*
$N=ESP32-03,T=22.4,H=56,D=151*
$N=ESP32-01,T=24.2,H=73,D=131*
$N=ESP32-02,T=32.2,H=58,D=220*
$N=ESP32-03,T=23.4,H=48,D=108*
$N=ESP32-01,T=25.4,H=67,D=148*

$N=ESP32-03,T=25.0,H=64,D=220*
$N=ESP32-01,T=27.2,H=58,D=125*
$N=ESP32-02,T=31.8,H=49,D=203*
$N=ESP32-03,T=23.3,H=44,D=127*
$N=ESP32-01,T=26.0,H=62,D=219*
$N=ESP32-02,T=30.3,H=60,D=175*
$N=ESP32-03,T=25.0,H=51,D=220*
$N=ESP32-01,T=25.9,H=69,D=220*
$N=ESP32-02,T=30.6,H=55,D=183*
$N=ESP32-03,T=23.9,H=60,D=124*
$N=ESP32-01,T=24.3,H=44,D=109*
$N=ESP32-02,T=29.7,H=55,D=126*
$N=ESP32-03,T=24.2,H=63,D=119*
$N=ESP32-01,T=27.3,H=73,D=133*
NESP32-02,T=32.9,H=70,D=129
$N=ESP32-03,T=22.6,H=47,D=109*
$N=ESP32-01,T=26.8,H=46,D=220*

$N=ESP32-03,T=22.4,H=43,D=220*
NESP32-01,T=27.3,H=75,D=220
$N=ESP32-02,T=30.7,H=60,D=122*
$N=ESP32-03,T=25.7,H=44,D=199*
$N=ESP32-01,T=25.9,H=65,D=220*
$N=ESP32-02,T=29.6,H=61,D=107*
$N=ESP32-03,T=23.2,H=54,D=220*
$N=ESP32-01,T=27.0,H=45,D=220*
$N=ESP32-02,T=30.9,H=44,D=115*
$N=ESP32-03,T=24.8,H=43,D=220*
$N=ESP32-01,T=26.0,H=41,D=220*
$N=ESP32-02,T=31.0,H=70,D=123*
$N=ESP32-03,T=22.6,H=43,D=112*
$N=ESP32-01,T=26.3,H=46,D=220*
$N=ESP32-02,T=32.1,H=54,D=220*
$N=ESP32-03,T=24.3,H=61,D=127*
$N=ESP32-01,T=26.6,H=64,D=163*
$N=ESP32-02,T=30.1,H=43,D=220*
$N=ESP32-03,T=25.2,H=56,D=198*
$N=ESP32-01,T=24.3,H=49,D=205*
$N=ESP32-02,T=32.2,H=63,D=220*
NESP32-03,T=23.9,H=46,D=169
$N=ESP32-01,T=27.4,H=74,D=220*
$N=ESP32-02,T=31.4,H=70,D=128*
$N=ESP32-03,T=25.1,H=68,D=220*
$N=ESP32-01,T=25.7,H=50,D=220*
$N=ESP32-02,T=2X.5,H=59,D=192*
$N=ESP32-03,T=24.4,H=44,D=220*"""
open("data/stream_boot.txt", "w", encoding="utf-8").write(_BOOT + "\n")
open("data/stream_day4.txt", "w", encoding="utf-8").write(_D4 + "\n")
BOOT_LINES = _BOOT.split("\n")

RAWS = np.array([512, 723, 1023, 100, 350, 890, 640, 210, 999, 480,
                 305, 760, 150, 555, 830, 420, 970, 60, 700, 260])
TEMPS24 = np.array([24.1, 26.7, 36.9, 28.3, 31.0, 22.8, 35.4, 29.5, 37.2, 25.0,
                    33.1, 21.4, 36.1, 27.7, 30.2, 23.9, 38.0, 26.1, 34.8, 28.9,
                    22.2, 35.9, 24.6, 31.7])

# stream_boot 의 정상 레코드 → 배열·레코드 목록 (2·4교시 재료)
_recs = []
for _l in BOOT_LINES:
    if not _l.strip(): continue
    try: _recs.append(parse_frame(_l))
    except (FrameError, ValueError): pass
BT = np.array([r[1] for r in _recs])                       # 온도 배열 (26,)
BN = np.array([r[0] for r in _recs])                       # 노드 배열 (26,)
RECORDS_BOOT = [{"node": r[0], "T": r[1], "H": r[2], "D": r[3]} for r in _recs]

# 이미지 실습용 배열 (파일 없이 생성)
_y, _x = np.mgrid[0:60, 0:80]
IMG = ((_x + _y) / (79 + 59)).astype(float)                # 흑백 그라디언트 (60, 80)
IMG_C = np.stack([_x / 79.0, _y / 59.0, np.ones_like(_x) * 0.3], axis=-1)  # 컬러 (60, 80, 3)

if not name: print("⚠️ 이름을 입력한 뒤 다시 실행하세요.")
else: print(f"준비 완료 — {name} · data/stream_boot.txt({len(BOOT_LINES)}줄) · data/stream_day4.txt(300줄)")

---
# 1교시 · NumPy 기초 — 벡터 연산·브로드캐스팅

🎬 **D4-1 브로드캐스팅·마스크** ①~④

### 따라하기

In [ ]:
a = np.array([1, 2, 3, 4])
print(a.shape, a.dtype)      # 배열을 받으면 shape 부터
print(a * 2)                 # 원소별 곱 — [1,2]*2 (리스트) 와 비교해 보세요
print(a > 2)                 # 비교도 배열로

import timeit
raws_list = list(RAWS) * 500
print("리스트 컴프리헨션:", round(timeit.timeit(lambda: [r/1023*3.3 for r in raws_list], number=100), 3), "초")
big = np.array(raws_list)
print("NumPy 벡터 연산 :", round(timeit.timeit(lambda: big/1023*3.3, number=100), 3), "초")

### TODO 1-1 · ADC → 전압 한 줄
`RAWS`(20개) 를 전압으로: `volts = np.round(RAWS / 1023 * 3.3, 2)`

In [ ]:
# TODO
raise NotImplementedError
print(volts[:5], volts.shape)

### TODO 1-2 · ALERT 건수
`TEMPS24` 에서 35 초과 개수를 `alert_count` 에 (int).

In [ ]:
# TODO: 마스크의 합
raise NotImplementedError
print(alert_count)

### TODO 1-3 · 섭씨 → 화씨
`fahr = TEMPS24 * 9/5 + 32` — 반복문 없이. 리스트였다면 몇 줄이었을지 주석으로.

In [ ]:
# TODO
raise NotImplementedError
print(np.round(fahr[:4], 1))

### TODO 1-4 · 브로드캐스팅 실험 (심화)
`M = np.arange(12).reshape(3, 4)` 에 `np.array([10, 20, 30, 40])` 을 더한 결과의 총합을 `bsum` 에. 이어서 `(3,)` 을 더해 보고 오류 메시지의 shapes 부분을 텍스트 셀에 옮겨 적으세요.

In [ ]:
# TODO
raise NotImplementedError
print(bsum)

**ValueError 의 shapes 부분:** 

In [ ]:
#@title ✅ 1교시 자동 채점
_check("1교시", [
    ("TODO 1-1 volts", lambda: volts.shape == (20,) and _h(list(np.round(volts, 2))) == "d5c9c8", "np.round(RAWS / 1023 * 3.3, 2)"),
    ("TODO 1-2 alert_count", lambda: int(alert_count) == 6, "(TEMPS24 > 35).sum()"),
    ("TODO 1-3 fahr", lambda: fahr.shape == (24,) and abs(fahr[2] - (36.9 * 9 / 5 + 32)) < 1e-9, "TEMPS24 * 9/5 + 32"),
    ("TODO 1-4 bsum", lambda: int(bsum) == 366, "(M + v).sum() — v 는 (4,)"),
])

In [ ]:
#@title 🔍 자기 점검 — np.array([1,2,3]) + np.array([1,2,3,4]) 의 결과는?
answer = "선택"  #@param ["선택", "[2,4,6,4]", "[2,4,6]", "ValueError (shapes (3,) (4,))", "0 이 채워진다"]
_quiz("1교시", "q1", answer, "ValueError (shapes (3,) (4,))",
      "뒤 축 3 vs 4 — 늘릴 수 없어 오류. shape 을 먼저 확인하는 습관. D4-1 ④.")

**오늘 막힌 점 (1교시):** 

---
# 2교시 · NumPy 활용 — 마스크·통계·이미지=배열

🎬 **D4-1** ⑤⑥⑦ (마스크) — 설정 셀이 만들어 둔 `BT`(온도 26개)·`BN`(노드)·`IMG`(흑백)·`IMG_C`(컬러) 사용

### 따라하기

In [ ]:
print(BT[:5], BT.shape)
print(BT[BT > 30])                    # 마스크 인덱싱
print(BT[BN == "ESP32-02"].mean())    # 노드 마스크 + 통계
plt.imshow(IMG, cmap="gray"); plt.title(f"IMG {IMG.shape}"); plt.show()

### TODO 2-1 · 노드별 평균 (마스크)
`avgs = {nid: round(float(BT[BN == nid].mean()), 2) for nid in ["ESP32-01", "ESP32-02", "ESP32-03"]}`

In [ ]:
# TODO
raise NotImplementedError
print(avgs)

### TODO 2-2 · 몇 건이, 언제
`n_alert = (BT > 35).sum()` (int) 와 `idx_max = BT.argmax()` (int).

In [ ]:
# TODO
raise NotImplementedError
print(n_alert, idx_max, BT[idx_max])

### TODO 2-3 · 이미지 = 배열
① `flipped = IMG[::-1]` 상하 반전 ② `dark = IMG * 0.5` ③ 둘을 imshow 로 확인.

In [ ]:
# TODO
raise NotImplementedError
fig, ax = plt.subplots(1, 3, figsize=(10, 2.6))
for a, im, t in zip(ax, [IMG, flipped, dark], ["원본", "상하 반전", "x0.5"]):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t); a.axis("off")
plt.show()

### TODO 2-4 · axis 통계 (심화)
`M2 = BT[:24].reshape(4, 6)` (4일 × 6회 온도라고 가정). `day_mean = M2.mean(axis=1)` 과 `slot_mean = M2.mean(axis=0)` 을 만들고, 각각의 의미를 한 줄로 텍스트 셀에.

In [ ]:
M2 = BT[:24].reshape(4, 6)
# TODO
raise NotImplementedError
print(np.round(day_mean, 2), day_mean.shape)
print(np.round(slot_mean, 2), slot_mean.shape)

**axis=1 / axis=0 의 의미:** 

In [ ]:
#@title ✅ 2교시 자동 채점
def _t21():
    ref = {nid: round(float(BT[BN == nid].mean()), 2) for nid in ["ESP32-01", "ESP32-02", "ESP32-03"]}
    return avgs == ref
def _t23():
    return flipped[0, 0] == IMG[-1, 0] and abs(dark.max() - IMG.max() * 0.5) < 1e-9
def _t24():
    return day_mean.shape == (4,) and slot_mean.shape == (6,) and abs(day_mean[0] - M2[0].mean()) < 1e-9
_check("2교시", [
    ("TODO 2-1 avgs", _t21, "BT[BN == nid].mean() 마스크 조합"),
    ("TODO 2-2 n_alert·idx_max", lambda: int(n_alert) == int((BT > 35).sum()) and int(idx_max) == int(BT.argmax()), "마스크 합 · argmax"),
    ("TODO 2-3 flipped·dark", _t23, "IMG[::-1] · IMG * 0.5"),
    ("TODO 2-4 axis", _t24, "axis 는 사라지는 방향"),
])

In [ ]:
#@title 🔍 자기 점검 — NumPy 슬라이스 b = a[2:5] 를 수정하면?
answer = "선택"  #@param ["선택", "a 는 안전하다", "a 도 바뀐다 (뷰)", "오류", "b 가 리스트가 된다"]
_quiz("2교시", "q1", answer, "a 도 바뀐다 (뷰)",
      "NumPy 슬라이스는 뷰(참조) — Day 2 참조 모델의 재림. 독립 사본은 .copy().")

**오늘 막힌 점 (2교시):** 

---
# 3교시 · Matplotlib — 시계열·히스토그램

읽히는 그래프 4요소: **title / xlabel / ylabel / grid** (+범례). savefig 는 show **앞에**.

### 따라하기

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(BT, marker="o", label="T")
plt.axhline(35, color="red", linestyle="--", label="ALERT 35C")
plt.title(f"stream_boot temperatures ({len(BT)} readings)")
plt.xlabel("sample #"); plt.ylabel("Temp (C)")
plt.grid(True); plt.legend()
plt.show()

### TODO 3-1 · 시계열 + ALERT scatter → 저장
위 그래프에 `mask = BT > 35` 인 점만 빨간 scatter 로 겹치고 `out/timeline_boot.png` 로 저장하세요 (savefig 를 show 앞에).

In [ ]:
# TODO
raise NotImplementedError
print(os.path.getsize("out/timeline_boot.png"), "bytes")

### TODO 3-2 · 히스토그램
`plt.hist(BT, bins=12)` + 4요소 → `out/hist_boot.png` 저장. 그래프만 보고 "정상 범위" 한 줄 코멘트를 텍스트 셀에.

In [ ]:
# TODO
raise NotImplementedError
print(os.path.exists("out/hist_boot.png"))

**정상 범위 코멘트:** 

### TODO 3-3 · 노드별 평균 막대 + 값 표시
2교시 `avgs` 로 막대 그래프, `plt.text` 로 막대 위 값 → `out/avg_by_node.png`.

In [ ]:
# TODO
raise NotImplementedError
print(os.path.exists("out/avg_by_node.png"))

### TODO 3-4 · 나란히 (심화)
`fig, ax = plt.subplots(1, 2, ...)` 로 시계열·히스토그램을 한 장에 → `out/combo.png`.

In [ ]:
# TODO
raise NotImplementedError
print(os.path.exists("out/combo.png"))

In [ ]:
#@title ✅ 3교시 자동 채점
def _png(p): return os.path.exists(p) and os.path.getsize(p) > 1000
_check("3교시", [
    ("TODO 3-1 timeline_boot.png", lambda: _png("out/timeline_boot.png"), "savefig 를 show 앞에"),
    ("TODO 3-2 hist_boot.png", lambda: _png("out/hist_boot.png"), "plt.hist(BT, bins=12)"),
    ("TODO 3-3 avg_by_node.png", lambda: _png("out/avg_by_node.png"), "plt.bar(avgs.keys(), avgs.values())"),
    ("TODO 3-4 combo.png", lambda: _png("out/combo.png"), "plt.subplots(1, 2)"),
])
print("(제목·축 라벨은 강사가 순회하며 확인합니다)")

In [ ]:
#@title 🔍 자기 점검 — png 가 0바이트가 되는 흔한 원인은?
answer = "선택"  #@param ["선택", "dpi 부족", "show() 후에 savefig()", "제목 누락", "색 미지정"]
_quiz("3교시", "q1", answer, "show() 후에 savefig()",
      "show 가 그림을 비운 뒤 저장하면 빈 파일 — savefig 먼저.")

**오늘 막힌 점 (3교시):** 

---
# 4교시 · pandas — DataFrame·groupby·CSV

🎬 **D4-2 DataFrame·groupby 흐름** ①~⑥ — 설정 셀의 `RECORDS_BOOT`(dict 26개) 사용

### 따라하기

In [ ]:
df = pd.DataFrame(RECORDS_BOOT)
print(df.head())
df.info()
print(df.describe().round(2))

### TODO 4-1 · DataFrame 만들기 3종 세트
`df = pd.DataFrame(RECORDS_BOOT)` — head/info/describe 를 실행해 보고, `n_rows = len(df)` 저장.

In [ ]:
# TODO
raise NotImplementedError
print(n_rows, list(df.columns))

### TODO 4-2 · alert 열 + 필터·정렬
① `df["alert"] = df["T"] > 35` ② `hot = df[df["alert"]].sort_values("T", ascending=False)`

In [ ]:
# TODO
raise NotImplementedError
print(hot[["node", "T"]])

### TODO 4-3 · groupby 요약 → CSV 왕복
`summary = df.groupby("node")["T"].agg(["mean", "max", "count"]).round(2)` → `out/sensor_summary_boot.csv` 저장(to_csv) → `back = pd.read_csv(...)` 로 복원.

In [ ]:
# TODO
raise NotImplementedError
print(summary)
print(back.shape)

### TODO 4-4 · 결측치 (심화)
제공된 `CSV_RAW` 를 `pd.read_csv(io.StringIO(CSV_RAW))` 로 읽어 ① `na_cnt = df2["T"].isna().sum()` ② `clean = df2.dropna()` — 전후 행 수 비교.

In [ ]:
CSV_RAW = """node,T,H
E1,24.1,50
E2,,70
E1,28.3,58
E3,,45
E2,31.0,66"""
# TODO
raise NotImplementedError
print(na_cnt, len(df2), "→", len(clean))

In [ ]:
#@title ✅ 4교시 자동 채점
def _t43():
    ref = pd.DataFrame(RECORDS_BOOT).groupby("node")["T"].agg(["mean", "max", "count"]).round(2)
    ok = summary.equals(ref) and os.path.exists("out/sensor_summary_boot.csv")
    return ok and "node" in back.columns and len(back) == 3
_check("4교시", [
    ("TODO 4-1 df 26행 4열", lambda: n_rows == 26 and set(df.columns) >= {"node", "T", "H", "D"}, "pd.DataFrame(RECORDS_BOOT)"),
    ("TODO 4-2 alert·hot", lambda: df["alert"].dtype == bool and len(hot) == int((df["T"] > 35).sum()) and len(hot) > 0 and hot.iloc[0]["T"] == df["T"].max(), "불리언 열 + 필터 + 내림차순"),
    ("TODO 4-3 groupby 왕복", _t43, "agg 리스트 + to_csv → read_csv"),
    ("TODO 4-4 결측", lambda: int(na_cnt) == 2 and len(clean) == 3, "isna().sum() · dropna()"),
])

In [ ]:
#@title 🔍 자기 점검 — df["T"] > 35 가 전부 False 인 흔한 원인은?
answer = "선택"  #@param ["선택", "pandas 버그", "T 가 문자열(object)", "35 가 커서", "정렬 안 함"]
_quiz("4교시", "q1", answer, "T 가 문자열(object)",
      "파싱 때 float 변환 누락 — df.info() 로 dtype 확인. \"25.3\" > 35 는 성립하지 않습니다.")

**오늘 막힌 점 (4교시):** 

---
# 5교시 · 파이썬 문법 마무리 — 제너레이터·이터레이터

🎬 **D3-5** 다시 보기 — "스트림은 본질적으로 제너레이터, readline 이 yield 였던 것"

### 따라하기

In [ ]:
# 이터레이터는 한 번 소모 (Day 2 map 회수)
m = map(str, [1, 2])
print(list(m), list(m))        # ['1', '2'] []

# zip — 쌍 순회
for n, t in zip(["E1", "E2"], [24.1, 36.9]):
    print(f"{n:6s}{t:6.1f}")

### TODO 5-1 · 손상 줄 번호 수집 (enumerate)
`BOOT_LINES` 를 `enumerate(..., 1)` 로 돌며 **비어 있지 않은데 parse_frame 이 실패하는** 줄 번호를 `bad_lines` 리스트에. (정답: 3개)

In [ ]:
bad_lines = []
# TODO
raise NotImplementedError
print(bad_lines)

### TODO 5-2 · read_frames 제너레이터
`read_frames(path)` — 파일을 열어 빈 줄은 건너뛰고, parse_frame 성공만 `yield`, 실패는 조용히 continue. 완성 후 **한 번 더 소모하면 0** 이 나오는 것을 확인하고 이유를 한 줄로.

In [ ]:
# TODO: def read_frames(path): ... yield ...
raise NotImplementedError

frames = read_frames("data/stream_boot.txt")
first = next(frames)
rest = sum(1 for _ in frames)
again = sum(1 for _ in frames)
print(first, "| 나머지", rest, "| 재소모", again)

**재소모가 0 인 이유:** 

### TODO 5-3 · f-string 정렬 표
`line1 = f"{'ESP32-01':10s}{26.43:8.2f}{12:7d}"` 형식으로 헤더 `head = f"{'node':10s}{'avg_T':>8s}{'count':>7s}"` 와 함께 출력.

In [ ]:
# TODO
raise NotImplementedError
print(head); print(line1)

### TODO 5-4 · 리스트 없는 평균 (심화)
제너레이터 표현식으로 `avg_gen = round(sum(t for _, t, _, _ in read_frames("data/stream_boot.txt")) / 26, 2)` + `parse_frame` 에 타입 힌트·docstring 이 이미 있는지 살펴보기.

In [ ]:
# TODO
raise NotImplementedError
print(avg_gen)

In [ ]:
#@title ✅ 5교시 자동 채점
_check("5교시", [
    ("TODO 5-1 bad_lines", lambda: bad_lines == [16, 19, 28], "빈 줄은 제외, 실패만 수집"),
    ("TODO 5-2 read_frames", lambda: inspect.isgeneratorfunction(read_frames) and first == ('ESP32-01', 26.2, 69, 220) and rest == 25 and again == 0, "yield · 한 번 소모"),
    ("TODO 5-3 f-string 표", lambda: line1 == "ESP32-01     26.43     12" and head.startswith("node"), "폭 10s / 8.2f / 7d"),
    ("TODO 5-4 avg_gen", lambda: avg_gen == 27.62, "대괄호 없는 제너레이터 표현식"),
])

In [ ]:
#@title 🔍 자기 점검 — 제너레이터를 한 번 소모한 뒤 다시 for 를 돌리면?
answer = "선택"  #@param ["선택", "처음부터 다시", "아무것도 나오지 않는다", "오류", "절반만"]
_quiz("5교시", "q1", answer, "아무것도 나오지 않는다",
      "이터레이터는 한 번 흘러가면 끝 — 다시 쓰려면 read_frames(path) 를 새로 호출.")

**오늘 막힌 점 (5교시):** 

---
# 6교시 · 미니 프로젝트 — 필수 기능

**"센서 데이터 수집·시각화 도구"** — `data/stream_day4.txt` 300줄
수신(read_frames) → 집계 → DataFrame → groupby 요약 → CSV → 그래프. 준실시간 갱신은 필수 2(수기 확인).

In [ ]:
#@title 필수 1 · 수신·집계 — collect()
def collect(path):
    """스트림 전체를 처리해 (records, good, bad, skip) 반환. records 는 dict 목록."""
    records, good, bad, skip = [], 0, 0, 0
    # TODO: BOOT 실습의 루프를 함수로 — 빈 줄 skip / parse_frame 성공 good / 실패 bad
    raise NotImplementedError
    return records, good, bad, skip

records, good, bad, skip = collect("data/stream_day4.txt")
print(good, bad, skip, "| records:", len(records))

In [ ]:
#@title 필수 2 · 준실시간 그래프 (시연용 — 강사 수기 확인)
# 50건마다 clear_output(wait=True) 후 최신 200건을 다시 그리는 데모.
# TODO: read_frames("data/stream_day4.txt") 를 돌며 temps 에 추가,
#       50건마다: clear_output(wait=True) → plt.plot(temps[-200:]) + 임계선 → plt.show()
raise NotImplementedError

In [ ]:
#@title 필수 3 · DataFrame·요약
# TODO: dfp = pd.DataFrame(records); dfp["alert"] = dfp["T"] > 35
#       summary300 = dfp.groupby("node")["T"].agg(["mean","max","count"]).round(2)
raise NotImplementedError
print(summary300)

In [ ]:
#@title 필수 4 · CSV 저장·왕복
# TODO: dfp.to_csv("out/readings.csv", index=False)
#       summary300.to_csv("out/sensor_summary.csv")
#       back300 = pd.read_csv("out/readings.csv") — 행 수 일치 확인
raise NotImplementedError
print(len(back300))

In [ ]:
#@title 필수 5 · 최종 그래프 저장
# TODO: timeline.png — 3노드 각각 plt.plot(label=노드) + 임계선 + 제목(건수 포함)·축 라벨·범례
#       hist.png — 전체 T 히스토그램(bins=20)
raise NotImplementedError
print(os.path.getsize("out/timeline.png"), os.path.getsize("out/hist.png"))

In [ ]:
#@title ✅ 6교시 프로젝트 필수 자동 채점
def _png(p): return os.path.exists(p) and os.path.getsize(p) > 1000
def _t63():
    ref = {"mean": dict(), "max": dict(), "count": dict()}
    exp = {"ESP32-01": {"mean": 25.74, "max": 27.7, "count": 87}, "ESP32-02": {"mean": 32.34, "max": 40.8, "count": 89}, "ESP32-03": {"mean": 23.99, "max": 25.9, "count": 90}}
    for nid, s in exp.items():
        if round(float(summary300.loc[nid, "mean"]), 2) != s["mean"]: return False
        if float(summary300.loc[nid, "max"]) != s["max"]: return False
        if int(summary300.loc[nid, "count"]) != s["count"]: return False
    return bool(dfp["alert"].dtype == bool)
_check("6교시-필수", [
    ("필수 1 수신·집계 (good/bad/skip)", lambda: (good, bad, skip) == (266, 23, 11) and len(records) == 266, "빈 줄 skip 분기가 try 앞에"),
    ("필수 3 groupby 요약 정합", _t63, "노드별 mean/max/count — dtype 은 df.info() 로"),
    ("필수 4 CSV 왕복", lambda: os.path.exists("out/sensor_summary.csv") and len(back300) == 266, "index=False 로 저장 후 read_csv"),
    ("필수 5 그래프 2종", lambda: _png("out/timeline.png") and _png("out/hist.png"), "savefig 를 show 앞에"),
])
print("(필수 2 준실시간과 그래프 라벨은 강사가 시연·순회로 확인합니다)")

**막힌 점 (6교시):** 

---
# 7교시 · 확장 + 시연 준비 (선택)

확장 메뉴 E1~E6 중 팀당 1개 이상. 아래 두 개는 자동 채점을 제공합니다 — 나머지는 결과물로 시연.

In [ ]:
#@title 확장 E2 · 손상 유형 리포트
err_counts = Counter()
# TODO: stream_day4 를 다시 돌며 FrameError / ValueError 를 이름별로 집계
#       (빈 줄은 집계하지 않음)
raise NotImplementedError
print(dict(err_counts))

In [ ]:
#@title 확장 E3 · 이동 평균 (rolling)
# TODO: dfp 에서 ESP32-02 만 골라 t2 = ...["T"].reset_index(drop=True)
#       roll_last = round(float(t2.rolling(5).mean().iloc[-1]), 2)
raise NotImplementedError
print(roll_last)

In [ ]:
#@title ✅ 7교시 확장 자동 채점 (선택)
_check("7교시-확장", [
    ("E2 err_counts", lambda: err_counts.get("FrameError", 0) == 15 and err_counts.get("ValueError", 0) == 8, "except 를 나눠 type(e).__name__ 집계"),
    ("E3 roll_last", lambda: roll_last == 31.36, "rolling(5).mean() 의 마지막 값"),
])

**시연 체크리스트 (3분)**
- [ ] 준실시간 수집 데모 1분 (리허설 1회 완료)
- [ ] 요약표·그래프 설명 1분 — "어느 노드가 언제 문제였나"
- [ ] 코드 하이라이트 + 배운 것 1분

---
# 8교시 · 시연·제출

- 팀 산출물: `out/` 4종 + 코드 → Drive `Day4/제출/팀이름/`
- 개인: 아래 제출 요약 실행 후 `Day4_이름.ipynb` 제출

In [ ]:
#@title 📋 제출 요약
ws_codes = ""  #@param {type:"string"}
#@markdown Day 4 개념 워크시트 결과 코드 7개 (쉼표 구분)

print(f"이름: {name}")
print("─" * 44)
total_ok = total_n = 0
for sec in ["1교시", "2교시", "3교시", "4교시", "5교시", "6교시-필수", "7교시-확장"]:
    ok, n = SCORES.get(sec, (0, 0))
    total_ok += ok; total_n += n
    print(f"{sec:10s} 자동 채점 {ok}/{n}" + ("  (미실행)" if n == 0 else ""))
print("─" * 44)
print(f"합계 {total_ok}/{total_n}")
quiz = {k: v for k, v in SCORES.items() if k.endswith("-quiz")}
print("자기 점검:", ", ".join(f"{k[:-5]} {sum(v.values())}/{len(v)}" for k, v in quiz.items()) or "없음")
print("개념 워크시트 코드:", ws_codes or "미입력")
outs = ["out/readings.csv", "out/sensor_summary.csv", "out/timeline.png", "out/hist.png"]
for p in outs:
    print(p + ":", "있음 ✅" if os.path.exists(p) else "없음 ❌")

**4일 중 가장 도움이 된 자료와 이유 (한 줄):** 